The kalman filter needs the motion model (aka state transition model) F.

First, define the motion model of the ship

In [1]:
import numpy as np

In [2]:
s = np.ones(4, dtype=float)[:, np.newaxis]
s.shape

(4, 1)

## Constant velocity linear propagation model
$$
F =
\begin{bmatrix}
1 \; 0 \; \Delta t \; 0 \\
0 \; 1 \; 0 \; \Delta t \\
0 \; 0 \; 1 \; 0 \\
0 \; 0 \; 0 \; 1
\end{bmatrix}
\;\;
s = 
\begin{bmatrix}
x \\
y \\
\dot{x} \\
\dot{y}
\end{bmatrix}
$$

In [3]:
def make_F_matrix(delta_t):
    F = np.array([
        [1, 0, delta_t, 0],
        [0, 1, 0, delta_t],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
        
    ], dtype=float)
    return F

In [4]:
F = make_F_matrix(10)
F

array([[ 1.,  0., 10.,  0.],
       [ 0.,  1.,  0., 10.],
       [ 0.,  0.,  1.,  0.],
       [ 0.,  0.,  0.,  1.]])

In [5]:
P = np.eye(4) * 0.25
P

array([[0.25, 0.  , 0.  , 0.  ],
       [0.  , 0.25, 0.  , 0.  ],
       [0.  , 0.  , 0.25, 0.  ],
       [0.  , 0.  , 0.  , 0.25]])

In [6]:
F@s

array([[11.],
       [11.],
       [ 1.],
       [ 1.]])

In [7]:
F@P@F.T

array([[25.25,  0.  ,  2.5 ,  0.  ],
       [ 0.  , 25.25,  0.  ,  2.5 ],
       [ 2.5 ,  0.  ,  0.25,  0.  ],
       [ 0.  ,  2.5 ,  0.  ,  0.25]])

## Nonlinear propagation model with unknown control inputs.

The state is represented by position $(x, y)$, a linear_velocity $s$, and a heading $\theta$. There is an unknown control vector $\pmb{u}$ consisting of acceleration $a$ and rotational velocity $\omega$; we assume these are normally distributed about 0 (most likely to not accelerate and not turn). $\sigma_\omega$ is based on the minimum/maximum turning radius.

$$
\pmb{x} = 
\begin{bmatrix}
x \\
y \\
s \\
\theta
\end{bmatrix}
\;\;
\pmb{u} = 
\begin{bmatrix}
a \sim \mathcal{N}(0,\,\sigma_a^{2}) \\
\omega \sim \mathcal{N}(0,\,\sigma_\omega^{2})
\end{bmatrix}
\\
x' = x + \cos{(\theta)}(s \Delta t + \frac{1}{2} a \Delta t ^2) \\
y' = y + \sin{(\theta)}(s \Delta t + \frac{1}{2} a \Delta t ^2) \\
s' = s + a \Delta t \\
\theta' = \theta + \omega \Delta t
$$